# II - Extraction tryptiques depuis un fichier `.conllu`

Ce notebook :
- lit un fichier `.conllu`
- extrait pour chaque verbe :
  - la **phrase**
  - le **sujet**
  - le **verbe**
  - l’**objet / complément**
  - les **dépendances** associées
- exporte le résultat en **CSV**


In [1]:
from pathlib import Path
import pandas as pd

## 1) Paramètres

Adaptez si besoin le chemin du `.conllu` et le chemin de sortie du `.csv`.


In [2]:
INPUT_CONLLU = "../results/from_sacr/all_annots.sacr.conllu"
OUTPUT_CSV  = "../results/csv_triptyques/triptyques_chap1to5_sacr.csv"

Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)

print("INPUT_CONLLU :", Path(INPUT_CONLLU).resolve())
print("OUTPUT_CSV   :", Path(OUTPUT_CSV).resolve())


INPUT_CONLLU : C:\Users\arnod\stage_lattice\Model_TDM\results\from_sacr\all_annots.sacr.conllu
OUTPUT_CSV   : C:\Users\arnod\stage_lattice\Model_TDM\results\csv_triptyques\triptyques_chap1to5_sacr.csv


## 2) Fonctions

Règles utilisées (reprises de tes notebooks) :
- **Sujet direct** : `nsubj`, `nsubj:pass`
- **Héritage du sujet** si le verbe est en `xcomp`, `ccomp`, `advcl`
- **Relative** `acl:relcl` : on prend le nom tête comme sujet implicite
- **Coordination** `conj` : on hérite du sujet du verbe gouverneur
- **Objets / compléments** : `obl:arg`, `obl:mod`, `obl:agent`, `obj`, `iobj`, `expl:comp`
- **Verbe prédicat** : on remonte si la tête est en `xcomp`, `ccomp`, `conj`, `advcl`


In [ ]:
def parse_feats(feats_str):
    if not feats_str or feats_str == "_":
        return {}
    out = {}
    for part in feats_str.split("|"):
        if "=" in part:
            k, v = part.split("=", 1)
            out[k] = v
    return out


def parse_custom_conllu(path):
    """ 
    Entrée : (str) un chemin de fichier .conllu
    Sortie : (dict) un dictionnaire avec toutes les informations du conllu
    """
    sentences = []
    comments = []
    tokens = []

    with open(path, "r", encoding="utf-8") as file:
        for raw_line in file:

            line = raw_line.rstrip("\n")

            # si on tombe sur une ligne vide, on ajoute les infos a sentences et on repart de 0
            if not line.strip():
                if tokens:
                    sentences.append({"comments": comments, "tokens": tokens}) 
                comments, tokens = [], []
                continue

            if line.startswith("#"):
                comments.append(line) # dans un CONLLU les commentaires commencent par un #, c'est la ou l'on note les infos de phrases plus globales
                continue

            parts = line.split("\t") # une ligne valide de conllu doit avoir 10 "parties" séparés d'une tabulation (\t)

            
            
            tok = {
                "id": int(parts[0]), # ID du token
                "form": parts[1], # forme du mot
                "lemma": parts[2], # lemme
                "upos": parts[3], # POS tag
                "xpos": parts[4], 
                "feats_raw": parts[5],
                "feats": parse_feats(parts[5]), 
                "head": int(parts[6]) if parts[6] != "_" else 0, # head syntaxique, si jamais il n'y en a pas on assigne la valeur 0
                "deprel": parts[7], # relation de dépendance (spacy)
                "col8": parts[8],   # annotation Propp / coréference
                "col9": parts[9],   # éntités nommées Propp : PER, FAC, LOC, etc.
            }
            
            tokens.append(tok)

    sentences.append({"comments": comments, "tokens": tokens})

    return sentences


PUNCT_NO_SPACE_BEFORE = {".", ",", ";", ":", "!", "?", ")", "]", "}", "%", "»", "…"}
PUNCT_NO_SPACE_AFTER = {"(", "[", "{", "«"}


def join_sentence_forms(tokens):
    forms = []
    for tok in tokens:
        f = tok.get("form", "")
        if not f:
            continue

        if tok.get("upos") == "PUNCT" and forms:
            forms[-1] = forms[-1] + f
        elif forms and forms[-1] in PUNCT_NO_SPACE_AFTER:
            forms[-1] = forms[-1] + f
        else:
            forms.append(f)

    return " ".join(forms).strip()


def token_by_id(tokens):
    return {tok["id"]: tok for tok in tokens}


def children_map(tokens):
    children = {}
    for tok in tokens:
        children.setdefault(tok["head"], []).append(tok)
    return children


def build_group(head_id, tokens):
    """
    Construit un groupe autour d’un token tête.
    On garde la logique simple et robuste :
    - on prend les descendants non verbaux
    - on exclut la ponctuation
    - on réordonne par ID
    """
    by_id = token_by_id(tokens)
    children = children_map(tokens)
    collected = set()

    def rec(tok_id):
        if tok_id in collected or tok_id not in by_id:
            return

        tok = by_id[tok_id]
        if tok["upos"] == "VERB":
            return
        if tok["deprel"] == "punct":
            return

        collected.add(tok_id)

        for child in children.get(tok_id, []):
            if child["upos"] != "VERB" and child["deprel"] != "punct":
                rec(child["id"])

    rec(head_id)

    ordered = [by_id[i]["form"] for i in sorted(collected)]
    if not ordered:
        return "NaN"

    out = []
    for f in ordered:
        if f in PUNCT_NO_SPACE_BEFORE and out:
            out[-1] = out[-1] + f
        else:
            out.append(f)

    return " ".join(out).strip()


def get_comment_value(comments, prefix):
    for c in comments:
        if c.startswith(prefix):
            return c.split("=", 1)[1].strip()
    return "NaN"


def find_subject(verb, tokens):
    # Sujet direct
    for tok in tokens:
        if tok["head"] == verb["id"] and tok["deprel"] in ["nsubj", "nsubj:pass"]:
            return tok

    # Héritage xcomp / ccomp / advcl
    if verb["deprel"] in ["xcomp", "ccomp", "advcl"]:
        head_id = verb["head"]
        for tok in tokens:
            if tok["head"] == head_id and tok["deprel"] in ["nsubj", "nsubj:pass"]:
                return tok

    # Relative
    if verb["deprel"] == "acl:relcl":
        head_id = verb["head"]
        for tok in tokens:
            if tok["id"] == head_id and tok["upos"] in ["NOUN", "PROPN", "PRON"]:
                return tok

    # Coordination
    if verb["deprel"] == "conj":
        head_id = verb["head"]
        head_tok = next((tok for tok in tokens if tok["id"] == head_id), None)
        if head_tok is not None:
            return find_subject(head_tok, tokens)

    return None


def find_predicate_verb(head_id, tokens):
    # Tête directement verbale
    for tok in tokens:
        if tok["id"] == head_id and tok["upos"] == "VERB":
            return tok

    # Remontée si xcomp / ccomp / conj / advcl
    for tok in tokens:
        if tok["id"] == head_id:
            if tok["deprel"] in ["xcomp", "ccomp", "conj", "advcl"] and tok["head"] != 0:
                return find_predicate_verb(tok["head"], tokens)

    return None


def find_objects_for_verb(verb, tokens):
    targets = ["obl:arg", "obl:mod", "obl:agent", "obj", "iobj", "expl:comp"]
    objs = []

    for tok in tokens:
        if tok["deprel"] in targets:
            pred = find_predicate_verb(tok["head"], tokens)
            if pred is not None and pred["id"] == verb["id"]:
                objs.append(tok)

    objs.sort(key=lambda x: x["id"])
    return objs




## 3) Extraction et export CSV

In [ ]:
def build_svo_dataframe(input_conllu):
    sentences = parse_custom_conllu(input_conllu)
    rows = []

    for sent_index, sent in enumerate(sentences, start=1):
        tokens = sent["tokens"]
        phrase_txt = join_sentence_forms(tokens)
        num_phrase = get_comment_value(sent["comments"], "# Num phrase ")
        num_paragr = get_comment_value(sent["comments"], "# Num paragr ")

        for verb in tokens:
            if verb["upos"] != "VERB":
                continue

            subj = find_subject(verb, tokens)
            if subj is not None:
                sujet = build_group(subj["id"], tokens)
                id_sujet = subj["id"]
                dep_sujet = subj["deprel"]
            else:
                sujet = "NaN"
                id_sujet = pd.NA
                dep_sujet = "NaN"

            objs = find_objects_for_verb(verb, tokens)

            # Toujours au moins une ligne par verbe
            if not objs:
                rows.append({
                    "Phrase": phrase_txt,
                    "Sujet": sujet,
                    "Verbe": verb["form"],
                    "Objet": "NaN",

                    "Dep_sujet": dep_sujet,
                    "Dep_verbe": verb["deprel"],
                    "Dep_objet": "NaN",

                    "ID_sujet": id_sujet,
                    "ID_verbe": verb["id"],
                    "ID_objet": pd.NA,

                    "Lemme_verbe": verb["lemma"],
                    "Head_verbe": verb["head"],

                    "Annotation_verbe_col8": verb["col8"],
                    "Annotation_verbe_col9": verb["col9"],

                    "Num_phrase": num_phrase,
                    "Num_paragr": num_paragr,
                    "Sentence_index": sent_index,
                })
            else:
                for obj in objs:
                    rows.append({
                        "Phrase": phrase_txt,
                        "Sujet": sujet,
                        "Verbe": verb["form"],
                        "Objet": build_group(obj["id"], tokens),

                        "Dep_sujet": dep_sujet,
                        "Dep_verbe": verb["deprel"],
                        "Dep_objet": obj["deprel"],

                        "ID_sujet": id_sujet,
                        "ID_verbe": verb["id"],
                        "ID_objet": obj["id"],

                        "IDs_sujet": group_token_ids(id_sujet, tokens),
                        "IDs_objet": group_token_ids(object_id, tokens),

                        "Lemme_verbe": verb["lemma"],
                        "Head_verbe": verb["head"],

                        "Annotation_verbe_col8": verb["col8"],
                        "Annotation_verbe_col9": verb["col9"],

                        "Num_phrase": num_phrase,
                        "Num_paragr": num_paragr,
                        "Sentence_index": sent_index,
                    })

    return pd.DataFrame(rows), sentences


df_svo, sentences = build_svo_dataframe(INPUT_CONLLU)
df_svo.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("CSV créé :", OUTPUT_CSV)
df_svo.head(20)


CSV créé : ../results/csv_triptyques/triptyques_chap1to5_sacr.csv


,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,Lemme_verbe,Head_verbe,Annotation_verbe_col8,Annotation_verbe_col9,Num_phrase,Num_paragr,Sentence_index
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville - row Burlington Garden...,NaN,acl,obj,<NA>,8,10,porter,7,5,PER,0,0,1
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,nsubj,acl:relcl,obl:mod,23,24,26,mourir,20,_,_,0,0,1
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l' année 1872,nsubj:pass,root,obl:mod,7,30,3,habiter,0,20,PER,0,0,1
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,nsubj:pass,root,obl:agent,7,30,32,habiter,0,20,PER,0,0,1
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l' un des membres les plus singuliers et les p...,nsubj:pass,root,obl:mod,7,30,39,habiter,0,20,PER,0,0,1
5,"En l' année 1872, la maison portant le numéro ...",la maison,semblât,NaN,nsubj:pass,advcl,NaN,7,59,<NA>,sembler,30,_,_,0,0,1
6,"En l' année 1872, la maison portant le numéro ...",NaN,prendre,à tâche,NaN,xcomp,obl:arg,<NA>,60,62,prendre,59,_,_,0,0,1
7,"En l' année 1872, la maison portant le numéro ...",NaN,faire,NaN,NaN,acl,NaN,<NA>,66,<NA>,faire,62,_,_,0,0,1
8,"En l' année 1872, la maison portant le numéro ...",qui,pût,NaN,nsubj,acl:relcl,NaN,67,68,<NA>,pouvoir,62,_,_,0,0,1
9,"En l' année 1872, la maison portant le numéro ...",qui,attirer,l' attention,nsubj,xcomp,obj,67,69,71,attirer,68,_,_,0,0,1


## 4) Des tryptiques au mouvement

Objectif de cette partie : maintenant que l'on a les tryptiques SUJET/VERBE/OBJET d'un texte en .csv, réutilisons les entités nommés stockées dans le .conllu correspondant pour avoir des tryptiques PERSONNAGE/VERBE/LIEU (ou dans l'autre sens).

In [ ]:

def to_int_or_none(x):
    if pd.isna(x):
        return None
    try:
        return int(float(x))
    except:
        return None


def group_token_ids(head_id, tokens):
    head_id = to_int_or_none(head_id)
    if head_id is None:
        return []

    by_id = token_by_id(tokens)
    children = children_map(tokens)
    collected = set()

    def rec(tok_id):
        if tok_id in collected or tok_id not in by_id:
            return

        tok = by_id[tok_id]

        if tok["upos"] == "VERB":
            return
        if tok["deprel"] == "punct":
            return

        collected.add(tok_id)

        for child in children.get(tok_id, []):
            rec(child["id"])

    rec(head_id)
    return sorted(collected)


def unique_list(values):
    out = []
    for v in values:
        if pd.notna(v) and v != "_" and v not in out:
            out.append(v)
    return out


def entities_in_ids(tokens, ids, wanted_cats):
    ids = set(ids)
    texts = []
    cats = []

    for tok in tokens:
        if tok["id"] in ids and tok["col9"] in wanted_cats:
            texts.append(tok["col8"])
            cats.append(tok["col9"])

    return unique_list(texts), unique_list(cats)